In [1]:
import torch                        # import torch library
import torch.nn as nn               # import nn module for creating layers, models and functions
import numpy as np                  # import numpy library
import matplotlib
import matplotlib.pyplot as plt     # import pyplot library
from torch.autograd import Variable # variable from numpy to torch
import torchquad
from torchquad import GaussLegendre as gl
from torchquad import Trapezoid as tp

seed = 27
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # set GPU as processing device
print(device)

cuda:0


In [3]:
epochs        = 6000  # number of training epochs
col_p         = 50    # number of collocation points
hidden_size   = 50    # number of neurons per hidden layer
learning_rate = 0.001 # learning rate parameter

E   = 21e9            # Pa    (Young Modulus)
miu = 0.23            # adim. (Poisson's ratio)
L   = 3               # m     (Beam length)
c   = 0.5             # m     (Beam height)
t   = 1               # m     (Beam thickness)
q   = 10000           # N/m2  (Distributed load q)
I   = 2*t*(c**3)/3    # m4    (Beam inertia)

class NN(nn.Module):
    def __init__(self):
        super(NN, self).__init__()
        self.fc1 = nn.Linear(2, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, hidden_size)
        self.fc5 = nn.Linear(hidden_size, 1)

    def forward(self,x,y):
        # Concatenate the input features into a single tensor
        inputs = torch.cat([x, y], axis=1)
        phi    = torch.tanh(self.fc1(inputs))
        phi    = torch.tanh(self.fc2(phi))
        phi    = torch.tanh(self.fc3(phi))
        phi    = torch.tanh(self.fc4(phi))
        phi    = self.fc5(phi)
        return phi

In [4]:
### (2) Model
# Create the neural network and move it to the GPU if available
NN = NN().to(device)                                            # transfer net to GPU for processing
mse_cost_function = torch.nn.MSELoss()                          # mean squared error as loss function
optimizer = torch.optim.Adam(NN.parameters(), lr=learning_rate) # Adam optimizer for parameter updating

In [5]:
## PDE as loss function.
def f(x,y,NN):
    phi = NN(x,y) # the dependent variable w is given by the network based on independent variables x and y
    ## Based on our f = d4w/dx4 + 2*d4w/dx2*dy2 + d4w/dy4, we need d4w/dx4, d4w/dx2*dy2 and d4w/dy4
   
    # d4phi/dx4
    dphidx   = torch.autograd.grad(phi.sum(),     x,create_graph=True)[0]  # dw/dx
    d2phidx2 = torch.autograd.grad(dphidx.sum(),  x,create_graph=True)[0]  # d2w/dx2
    d3phidx3 = torch.autograd.grad(d2phidx2.sum(),x,create_graph=True)[0]  # d3w/dx3
    d4phidx4 = torch.autograd.grad(d3phidx3.sum(),x,create_graph=True)[0]  # d4w/dx4
    
    # d4phi/dy4
    dphidy   = torch.autograd.grad(phi.sum(),     y,create_graph=True)[0]  # dw/dy
    d2phidy2 = torch.autograd.grad(dphidy.sum(),  y,create_graph=True)[0]  # d2w/dy2
    d3phidy3 = torch.autograd.grad(d2phidy2.sum(),y,create_graph=True)[0]  # d3w/dy3
    d4phidy4 = torch.autograd.grad(d3phidy3.sum(),y,create_graph=True)[0]  # d4w/dy4
    
    # d4phi/dx2dy2
    d3phidx2dy  = torch.autograd.grad(d2phidx2.sum(),  y,create_graph=True)[0]  # d3w/dx2*dy
    d4phidx2dy2 = torch.autograd.grad(d3phidx2dy.sum(),y,create_graph=True)[0]  # d3w/dx2*dy2
    
    pde = d4phidx4 + 2*d4phidx2dy2 + d4phidy4
    return pde

In [6]:
## Data from Boundary Conditions
## BC gives us datapoints for training
## We will find out phi(x,y) for x in range [-L,L] and y in range [-c,c]

#################################### 1. Txy(x,y=+c,z) = 0 ####################################
x_bc1   =   np.random.uniform(low=-L, high=L, size=(col_p,1))
y_bc1   = c*np.ones((col_p,1))
txy_bc1 =   np.zeros((col_p,1))

pt_x_bc1   = Variable(torch.from_numpy(x_bc1).float(),requires_grad=True).to(device)
pt_y_bc1   = Variable(torch.from_numpy(y_bc1).float(),requires_grad=True).to(device)
pt_txy_bc1 = Variable(torch.from_numpy(txy_bc1).float(),requires_grad=False).to(device)

#################################### 2. Txy(x,y=-c,z) = 0 ####################################
x_bc2   =    np.random.uniform(low=-L, high=L, size=(col_p,1))
y_bc2   = -c*np.ones((col_p,1))
txy_bc2 =    np.zeros((col_p,1))  

pt_x_bc2   = Variable(torch.from_numpy(x_bc2).float(),requires_grad=True).to(device)
pt_y_bc2   = Variable(torch.from_numpy(y_bc2).float(),requires_grad=True).to(device)
pt_txy_bc2 = Variable(torch.from_numpy(txy_bc2).float(),requires_grad=False).to(device)

#################################### 3. Sy(x,y=+c,z) = q/t ####################################
x_bc3  =       np.random.uniform(low=-L, high=L, size=(col_p,1))
y_bc3  =     c*np.ones((col_p,1))
Sy_bc3 = (q/t)*np.ones((col_p,1))

pt_x_bc3  = Variable(torch.from_numpy(x_bc3).float(),requires_grad=True).to(device)
pt_y_bc3  = Variable(torch.from_numpy(y_bc3).float(),requires_grad=True).to(device)
pt_Sy_bc3 = Variable(torch.from_numpy(Sy_bc3).float(),requires_grad=False).to(device)

#################################### 4. Sy(x,y=-c,z) = 0 ####################################
x_bc4  =    np.random.uniform(low=-L, high=L, size=(col_p,1))
y_bc4  = -c*np.ones((col_p,1))
Sy_bc4 =    np.zeros((col_p,1))

pt_x_bc4  = Variable(torch.from_numpy(x_bc4).float(),requires_grad=True).to(device)
pt_y_bc4  = Variable(torch.from_numpy(y_bc4).float(),requires_grad=True).to(device)
pt_Sy_bc4 = Variable(torch.from_numpy(Sy_bc4).float(),requires_grad=False).to(device)

#################################### 5. V(+L) = qL ####################################
x_bc5 =     L*np.ones((col_p,1))
y_bc5 =       np.random.uniform(low=-c, high=c, size=(col_p,1))
V_bc5 = (q*L)*np.ones((col_p,1))

pt_x_bc5 = Variable(torch.from_numpy(x_bc5).float(),requires_grad=True).to(device)
pt_y_bc5 = Variable(torch.from_numpy(y_bc5).float(),requires_grad=True).to(device)
pt_V_bc5 = Variable(torch.from_numpy(V_bc5).float(),requires_grad=False).to(device)

#################################### 6. V(-L) = -qL ####################################
x_bc6 =     -L*np.ones((col_p,1))
y_bc6 =        np.random.uniform(low=-c, high=c, size=(col_p,1))
V_bc6 = -(q*L)*np.ones((col_p,1))

pt_x_bc6 = Variable(torch.from_numpy(x_bc6).float(),requires_grad=True).to(device)
pt_y_bc6 = Variable(torch.from_numpy(y_bc6).float(),requires_grad=True).to(device)
pt_V_bc6 = Variable(torch.from_numpy(V_bc6).float(),requires_grad=False).to(device)

#################################### 7. M(-L) = 0 ####################################
x_bc7 = -L*np.ones((col_p,1))
y_bc7 =    np.random.uniform(low=-c, high=c, size=(col_p,1))
M_bc7 =    np.zeros((col_p,1))

pt_x_bc7 = Variable(torch.from_numpy(x_bc7).float(),requires_grad=True).to(device)
pt_y_bc7 = Variable(torch.from_numpy(y_bc7).float(),requires_grad=True).to(device)
pt_M_bc7 = Variable(torch.from_numpy(M_bc7).float(),requires_grad=False).to(device)

#################################### 8. M(L) = 0 ####################################
x_bc8 = L*np.ones((col_p,1))
y_bc8 =   np.random.uniform(low=-c, high=c, size=(col_p,1))
M_bc8 =   np.zeros((col_p,1))

pt_x_bc8 = Variable(torch.from_numpy(x_bc8).float(),requires_grad=True).to(device)
pt_y_bc8 = Variable(torch.from_numpy(y_bc8).float(),requires_grad=True).to(device)
pt_M_bc8 = Variable(torch.from_numpy(M_bc8).float(),requires_grad=False).to(device)

#################################### 9. faxial = 0 ####################################
x_bc9      = np.random.uniform(low=-L, high=L, size=(col_p,1))
y_bc9      = np.random.uniform(low=-c, high=c, size=(col_p,1))
faxial_bc9 = np.zeros((col_p,1))

pt_x_bc9      = Variable(torch.from_numpy(x_bc9).float(),requires_grad=True).to(device)
pt_y_bc9      = Variable(torch.from_numpy(y_bc9).float(),requires_grad=True).to(device)
pt_faxial_bc9 = Variable(torch.from_numpy(faxial_bc9).float(),requires_grad=False).to(device)

In [7]:
### (3) Training / Fitting

Loss_bc1  = []
Loss_bc2  = []
Loss_bc3  = []
Loss_bc4  = []
Loss_bc5  = []
Loss_f    = []
Loss_track= []            # Initialize empty list for gathering loss values

NN.train()

for epoch in range(epochs):
    optimizer.zero_grad() # To make the gradients zero
    
    # Loss based on boundary conditions
    # BC 1.      
    NN_bc1_out   = NN(pt_x_bc1,pt_y_bc1) # output NN(x,y) --> phi
    dNNdx_bc1    = torch.autograd.grad(NN_bc1_out.sum(),pt_x_bc1,create_graph=True)[0]
    d2NNdxdy_bc1 = torch.autograd.grad(dNNdx_bc1.sum(),pt_y_bc1,create_graph=True)[0]
    mse_bc1      = mse_cost_function(-d2NNdxdy_bc1, pt_txy_bc1)
    
    # BC 2.    
    NN_bc2_out   = NN(pt_x_bc2,pt_y_bc2) # output NN(x,y) --> phi
    dNNdx_bc2    = torch.autograd.grad(NN_bc2_out.sum(),pt_x_bc2,create_graph=True)[0]
    d2NNdxdy_bc2 = torch.autograd.grad(dNNdx_bc2.sum(),pt_y_bc2,create_graph=True)[0]
    mse_bc2      = mse_cost_function(-d2NNdxdy_bc2, pt_txy_bc2)
    
    # BC 3.      
    NN_bc3_out   = NN(pt_x_bc3,pt_y_bc3) # output NN(x,y) --> phi
    dNNdx_bc3    = torch.autograd.grad(NN_bc3_out.sum(),pt_x_bc3,create_graph=True)[0]
    d2NNdx2_bc3  = torch.autograd.grad(dNNdx_bc3.sum(),pt_x_bc3,create_graph=True)[0]
    mse_bc3      = mse_cost_function(d2NNdx2_bc3, pt_Sy_bc3)

    # BC 4.      
    NN_bc4_out   = NN(pt_x_bc4,pt_y_bc4) # output NN(x,y) --> phi
    dNNdx_bc4    = torch.autograd.grad(NN_bc4_out.sum(),pt_x_bc4,create_graph=True)[0]
    d2NNdx2_bc4  = torch.autograd.grad(dNNdx_bc4.sum(),pt_x_bc4,create_graph=True)[0]
    mse_bc4      = mse_cost_function(d2NNdx2_bc4, pt_Sy_bc4)

    #BC 5.
    NN_bc5_out   = NN(pt_x_bc5,pt_y_bc5) # output NN(x,y) --> phi
    dNNdx_bc5    = torch.autograd.grad(NN_bc5_out.sum(),pt_x_bc5,create_graph=True)[0]
    d2NNdxdy_bc5 = torch.autograd.grad(dNNdx_bc5.sum(),pt_y_bc5,create_graph=True)[0]
    integral     = torch.trapezoid(d2NNdxdy_bc5)
    #integral     = gl.integrate(d2NNdxdy_bc5, dim=2, N=10001, integration_domain=[[-c,c],[0,t]])
    mse_bc5      = mse_cost_function(integral, pt_V_bc5)
     
    # Loss based on PDE
    #x_col       = np.random.rand(col_p,1)   
    x_col        = np.random.uniform(low=-L, high=L, size=(col_p,1))
    y_col        = np.random.uniform(low=-c, high=c, size=(col_p,1))
    all_zeros    = np.zeros((col_p,1))
    
    pt_x_col     = Variable(torch.from_numpy(x_col).float(), requires_grad=True).to(device)
    pt_y_col     = Variable(torch.from_numpy(y_col).float(), requires_grad=True).to(device)
    pt_all_zeros = Variable(torch.from_numpy(all_zeros).float(), requires_grad=False).to(device)
    
    f_out        = f(pt_x_col,pt_y_col,NN)                # Output of f(x)
    mse_f        = mse_cost_function(f_out, pt_all_zeros)
    
    # Combining the loss functions
    loss = mse_bc1 + mse_bc2 + mse_bc3 + mse_bc4 + mse_bc5 + mse_f
    
    loss.backward()  # This is for computing gradients using backward propagation
    optimizer.step() # This is equivalent to: theta_new = theta_old - alpha * derivative of J w.r.t theta
    
    loss_bc1 = mse_bc1.data.cpu().numpy()
    loss_bc2 = mse_bc2.data.cpu().numpy()
    loss_bc3 = mse_bc3.data.cpu().numpy()
    loss_bc4 = mse_bc4.data.cpu().numpy()
    loss_bc5 = mse_bc5.data.cpu().numpy()
    loss_f   =   mse_f.data.cpu().numpy()
    loss_np  =    loss.data.cpu().numpy()          # Torch tensors to Numpy
    
    Loss_bc1   = np.append(Loss_bc1, loss_bc1)
    Loss_bc2   = np.append(Loss_bc2, loss_bc2)
    Loss_bc3   = np.append(Loss_bc3, loss_bc3)
    Loss_bc4   = np.append(Loss_bc4, loss_bc4)
    Loss_bc5   = np.append(Loss_bc5, loss_bc5)
    Loss_f     = np.append(Loss_f,    loss_f)
    Loss_track = np.append(Loss_track,loss_np)   # Tracking loss values and creating a list
    
    with torch.autograd.no_grad():
        if epoch % 100 == 0: print(epoch,"Traning Loss:",loss.data) # Print loss every 10 epochs

C:\Users\anfel\anaconda3\Lib\site-packages\torch\nn\modules\loss.py:536: UserWarning: Using a target size (torch.Size([50, 1])) that is different to the input size (torch.Size([50])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


0 Traning Loss: tensor(1.0000e+09, device='cuda:0')
100 Traning Loss: tensor(9.9993e+08, device='cuda:0')
200 Traning Loss: tensor(9.9988e+08, device='cuda:0')
300 Traning Loss: tensor(9.9989e+08, device='cuda:0')
400 Traning Loss: tensor(9.9976e+08, device='cuda:0')
500 Traning Loss: tensor(9.9996e+08, device='cuda:0')
600 Traning Loss: tensor(9.9992e+08, device='cuda:0')
700 Traning Loss: tensor(9.9987e+08, device='cuda:0')
800 Traning Loss: tensor(9.9994e+08, device='cuda:0')
900 Traning Loss: tensor(9.9993e+08, device='cuda:0')
1000 Traning Loss: tensor(9.9994e+08, device='cuda:0')
1100 Traning Loss: tensor(9.9992e+08, device='cuda:0')
1200 Traning Loss: tensor(9.9991e+08, device='cuda:0')
1300 Traning Loss: tensor(9.9990e+08, device='cuda:0')
1400 Traning Loss: tensor(9.9987e+08, device='cuda:0')
1500 Traning Loss: tensor(9.9988e+08, device='cuda:0')
1600 Traning Loss: tensor(9.9986e+08, device='cuda:0')
1700 Traning Loss: tensor(9.9977e+08, device='cuda:0')


KeyboardInterrupt: 

In [ ]:
#Loss graphics
ep = np.linspace(1,epochs,epochs)
plt.semilogy(ep,Loss_track)
plt.title('MSE Loss vs epochs')
plt.xlabel('Epochs')
plt.ylabel('Total MSE loss')

plt.semilogy(ep,Loss_bc1)
plt.semilogy(ep,Loss_bc2)
plt.semilogy(ep,Loss_bc3)
plt.semilogy(ep,Loss_bc4)
plt.semilogy(ep,Loss_bc4)
plt.semilogy(ep,Loss_f)

plt.legend(['Total MSE Loss','BC1','BC2','BC3','BC4','BC5','PDE'])
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

x = np.linspace(0.0, a, col_p)
y = np.linspace(0.0, b, col_p)

X,Y = np.meshgrid(x,y)

x_reshape = x.reshape((col_p, 1))
y_reshape = y.reshape((col_p, 1))

pt_x = torch.from_numpy(x_reshape).float().to(device)
pt_y = torch.from_numpy(y_reshape).float().to(device)
pt_w_pred = NN(pt_x, pt_y)

w_pred = pt_w_pred.data.cpu().numpy()
w = np.squeeze(w_pred)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, w)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
import numpy as np

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

x    = np.arange(0.0, a, 0.1)
y    = np.arange(0.0, b, 0.1)
X,Y  = np.meshgrid(x,y)
x    = np.ravel(X).reshape(-1,1)
y    = np.ravel(Y).reshape(-1,1)

pt_x = Variable(torch.from_numpy(x).float(), requires_grad=True).to(device)
pt_y = Variable(torch.from_numpy(y).float(), requires_grad=True).to(device)
pt_w = NN(pt_x,pt_y)
w    = pt_w.data.cpu().numpy()
W    = w.reshape(X.shape)

surf = ax.plot_surface(X,Y,W)

ax.zaxis.set_major_locator(LinearLocator(10))
ax.zaxis.set_major_formatter(FormatStrFormatter('%.02f'))

fig.colorbar(surf, shrink=0.5, aspect=5)

plt.show()